# Deliverable 1 — Build & Verify the Benchmark

**Owners:** Question-set lead (runs this), everyone reviews.

1. Run `python scripts/setup_db.py` once from the repo root (builds `data/chinook.db`).
2. Add your 15–20 questions to `data/questions.json` (adapted from Spider — see README).
3. Run this notebook: it executes every gold query and saves the results to `data/gold_results.json`.
4. **A second teammate** eyeballs each result and initials it in the review table below.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.paths import *
pd.set_option("display.max_colwidth", 120)

In [ ]:
questions = json.loads(QUESTIONS.read_text())
print(f"{len(questions)} questions loaded")
pd.DataFrame(questions)[["id", "difficulty", "category", "question"]]

## Coverage check — the guide requires a mix (single-table, joins, aggregation, ≥1 subquery), mostly easy/medium

In [ ]:
df = pd.DataFrame(questions)
display(df["category"].value_counts(), df["difficulty"].value_counts())
assert 15 <= len(df) <= 20, f"Need 15–20 questions, have {len(df)}"
assert (df["category"] == "subquery").any(), "Need at least one subquery question"
assert df["id"].is_unique

## Run every gold query

In [ ]:
from src.grading import run_query

gold = {}
for q in questions:
    rows, err = run_query(NORM_DB, q["gold_sql"])
    if err:
        print(f"❌ {q['id']}: {err}")
        continue
    if not rows:
        print(f"⚠️  {q['id']}: returned 0 rows — is that really the right answer?")
    gold[q["id"]] = rows
    print(f"✅ {q['id']}: {len(rows)} rows  e.g. {rows[:3]}")

In [ ]:
out = DATA / "gold_results.json"
out.write_text(json.dumps({k: [list(r) for r in v] for k, v in gold.items()}, indent=1))
print(f"Saved {len(gold)}/{len(questions)} gold result sets -> {out.relative_to(ROOT)}")

## Peer review log
Edit this table as you verify each result (reviewer ≠ author).

| ID | Author | Reviewer | Result looks right? | Notes |
|----|--------|----------|---------------------|-------|
| Q01 |  |  |  |  |